# DDPM diffusion model for cat images

## Imports

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from scripts import (
    DataFixedConfig,
    DataGridConfig,
    DiffusionExperiment,
    DiffusionFixedConfig,
    DiffusionGridConfig,
    TrainFixedConfig,
    TrainGridConfig,
    build_dataloaders,
    build_diffusion,
    compute_fid,
    count_parameters,
    diffusion_sample_iterator,
    expand_experiment_grid,
    interpolate_diffusion,
    output_paths,
    prepare_data,
    report_fid,
    run_name,
    sample_diffusion_latents,
    save_history,
    save_history_plot,
    save_interpolation_grid,
    save_json,
    save_sample_grid,
    train_diffusion,
    update_total_stats,
)

## Parameters

In [2]:
# Training time on a free Colab T4 GPU at image_size=64 and ~14k cats:
#   batch_size=64, base_channels=64, T=400 timesteps -> ~75s per epoch.
#   30 epochs ~= 38 minutes training; another ~6 minutes per FID computation
#   because reverse sampling at T=400 dominates wall-clock time.
# Reduce T to 200 to halve sampling cost at modest quality loss.

EXPERIMENT_NAME = "diffusion"

# --- Data ---------------------------------------------------------------
DATA_DIR = "../data/cats-faces"
OUTPUT_DIR = "../reports/runs"
CACHE_DIR = "../.cache/fid"
IMAGE_SIZE = 64
TRAIN_FRACTION = 0.95
VALIDATION_FRACTION = 0.05
AUGMENT_FLIP = False
GRAYSCALE = False      # True collapses RGB to 1 channel (channels auto-set below)
POSTERIZE_BITS = 6  # None disables; e.g. 4 keeps 16 levels/channel, 2 keeps 4

# --- UNet / scheduler ---------------------------------------------------
TIMESTEPS = 400                # 200..500 is the practical Colab range
SCHEDULE = "cosine"            # "linear" | "cosine"
BASE_CHANNELS = 64
CHANNEL_MULTS = (1, 2, 2, 2)   # 4 resolutions: 64 -> 32 -> 16 -> 8
TIME_EMBEDDING_DIM = 128
BETA_START = 1e-4
BETA_END = 0.02
EMA_DECAY = 0.999

# --- Optimization -------------------------------------------------------
LEARNING_RATE = 2e-4
EPOCHS = 30
BATCH_SIZE = 64
SEED = 42

# --- Early stopping -----------------------------------------------------
# Aggressive: stop unless validation_loss improves by at least MIN_DELTA
# within PATIENCE epochs. Sub-MIN_DELTA wiggles are treated as noise.
# Diffusion noise-prediction MSE is small (~0.01-0.1), so MIN_DELTA is too.
# Note: when restore_best is True it takes precedence over the EMA copy,
# since EMA weights are never validated.
EARLY_STOPPING = True
EARLY_STOPPING_PATIENCE = 8
EARLY_STOPPING_MIN_DELTA = 1e-4
EARLY_STOPPING_RESTORE_BEST = True

# --- Checkpointing ------------------------------------------------------
# Final checkpoint lands in the run's own checkpoints/ folder by default.
SAVE_CHECKPOINT = True  # Save model + optimizer + EMA shadow + history at end of training
RESUME_FROM = None      # Path to a .pt checkpoint to resume from, or None to start fresh

# --- Logging / sampling -------------------------------------------------
SAMPLE_EVERY = 10              # diffusion sampling is expensive; log sparingly
FID_REAL_SAMPLES = 1000
FID_FAKE_SAMPLES = 1000
DEVICE = "cuda"
PROGRESS_BACKEND = "terminal"

experiment = DiffusionExperiment(
    name=EXPERIMENT_NAME,
    data_fixed=DataFixedConfig(
        data_dir=DATA_DIR,
        cache_dir=CACHE_DIR,
        output_dir=OUTPUT_DIR,
        image_size=IMAGE_SIZE,
        channels=1 if GRAYSCALE else 3,
        grayscale=GRAYSCALE,
        posterize_bits=POSTERIZE_BITS,
    ),
    data_grid=DataGridConfig(
        train_fraction=TRAIN_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
        augment_flip=AUGMENT_FLIP,
        seed=SEED,
    ),
    train_fixed=TrainFixedConfig(
        device=DEVICE,
        progress_backend=PROGRESS_BACKEND,
        fid_real_samples=FID_REAL_SAMPLES,
        fid_fake_samples=FID_FAKE_SAMPLES,
        early_stopping=EARLY_STOPPING,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        early_stopping_min_delta=EARLY_STOPPING_MIN_DELTA,
        early_stopping_restore_best=EARLY_STOPPING_RESTORE_BEST,
    ),
    train_grid=TrainGridConfig(
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        sample_every=SAMPLE_EVERY,
    ),
    model_fixed=DiffusionFixedConfig(
        time_embedding_dim=TIME_EMBEDDING_DIM,
    ),
    model_grid=DiffusionGridConfig(
        timesteps=TIMESTEPS,
        schedule=SCHEDULE,
        base_channels=BASE_CHANNELS,
        channel_mults=CHANNEL_MULTS,
        learning_rate=LEARNING_RATE,
        beta_start=BETA_START,
        beta_end=BETA_END,
        ema_decay=EMA_DECAY,
    ),
)

runs = expand_experiment_grid(experiment)
run = runs[0]
print(f"Generated {len(runs)} configuration(s); the first is selected for this notebook run.")

Generated 1 configuration(s); the first is selected for this notebook run.


## Setup

In [3]:
torch.manual_seed(run["data"]["seed"])
np.random.seed(run["data"]["seed"])

run_dir = run_name(experiment.kind, run)
paths = output_paths(experiment.name, experiment.data_fixed.output_dir, run_dir)
save_json(paths["configs"] / "run.json", {"experiment": experiment.name, **run})
print(f"Run output: {paths['root']}")

prepared = prepare_data(
    experiment.data_fixed,
    train_fraction=run["data"]["train_fraction"],
    validation_fraction=run["data"]["validation_fraction"],
    augment_flip=run["data"]["augment_flip"],
    seed=run["data"]["seed"],
)
print(f"Train images: {len(prepared.train_dataset)} | Validation: {len(prepared.validation_dataset)}")

train_loader, validation_loader = build_dataloaders(
    prepared,
    experiment.data_fixed,
    batch_size=run["train"]["batch_size"],
    seed=run["data"]["seed"],
)

model = build_diffusion(run["model"], experiment.model_fixed, experiment.data_fixed)
print(f"Diffusion parameters: {count_parameters(model):,}")


Run output: ..\reports\runs\diffusion\20260607_143619_diff_t400_cosine_c64_lr0_0002_e30
Train images: 14960 | Validation: 787
Diffusion parameters: 8,177,155


## Training

In [ ]:
training_result = train_diffusion(
    model,
    train_loader,
    validation_loader,
    model_params=run["model"],
    train_params=run["train"],
    train_fixed=experiment.train_fixed,
    sample_dir=paths["samples"],
    checkpoint_path=paths["checkpoints"] / "diffusion_final.pt" if SAVE_CHECKPOINT else None,
    resume_from=RESUME_FROM,
)

history = training_result["history"]
save_history(history, paths["metrics"] / "history.csv")
save_history_plot(
    history,
    paths["figures"] / "loss_curves.png",
    metrics=["loss", "validation_loss"],
    title="Diffusion Loss Curves",
)
elapsed = training_result["elapsed_seconds"]
print(f"Trained in {int(elapsed // 3600):02d}:{int(elapsed % 3600 // 60):02d}:{int(elapsed % 60):02d} on {training_result['device']}.")
update_total_stats(
    experiment.data_fixed.output_dir,
    experiment.kind,
    run_dir,
    experiment.name,
    elapsed,
    training_result["device"],
)

Using device: cuda
Diffusion parameters: 8,177,155


Diffusion Training:   0%|          | 0/30 [00:00<?, ?it/s]

## Evaluation

In [ ]:
device = next(model.parameters()).device

with torch.no_grad():
    samples = model.sample(16, device)
    save_sample_grid(samples, paths["figures"] / "samples_final.png", nrow=4)

metrics_df = pd.DataFrame(history)
fig, axis = plt.subplots(figsize=(8, 5))
axis.plot(metrics_df["epoch"], metrics_df["loss"], label="train")
axis.plot(metrics_df["epoch"], metrics_df["validation_loss"], label="validation")
axis.set_title("Diffusion training loss"); axis.set_xlabel("Epoch")
axis.legend(); axis.grid(alpha=0.3)
plt.tight_layout(); plt.show()

sampler = diffusion_sample_iterator(
    model,
    num_samples=experiment.train_fixed.fid_fake_samples,
    batch_size=run["train"]["batch_size"],
    device=device,
    seed=run["data"]["seed"],
)
fid_start = time.perf_counter()
fid_score = compute_fid(
    sampler,
    experiment.data_fixed,
    num_real_samples=experiment.train_fixed.fid_real_samples,
    seed=run["data"]["seed"],
    device=device,
)
fid_elapsed = time.perf_counter() - fid_start
report_fid(fid_score, elapsed_seconds=fid_elapsed)
save_json(paths["metrics"] / "fid.json", {"fid": fid_score, "elapsed_seconds": fid_elapsed})

img = plt.imread(paths["figures"] / "samples_final.png")
plt.figure(figsize=(8, 8))
plt.imshow(img); plt.axis("off")
plt.title(f"Diffusion samples (FID={fid_score:.2f})")
plt.show()


## Interpolation

In [ ]:
z1, z2 = sample_diffusion_latents(model, device, run["data"]["seed"])
images = interpolate_diffusion(model, z1, z2, num_steps=10)
interpolation_path = paths["figures"] / "interpolation.png"
save_interpolation_grid(images, interpolation_path)

img = plt.imread(interpolation_path)
plt.figure(figsize=(20, 2.6))
plt.imshow(img); plt.axis("off")
plt.title("Diffusion latent interpolation: noise z1 -> noise z2 (10 steps)")
plt.show()
